In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns

# Step 1: Import Required Libraries

# Step 2: Load the MNIST Dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("Training set shape:", x_train.shape)
print("Test set shape:", x_test.shape)

# Step 3: Preprocess the Data
# Reshape to match CNN input: (samples, height, width, channels)
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
# Print sample after normalization
print("Normalized image shape:", x_train[0].shape)

def create_cnn_model(num_conv_layers=1, num_dense_layers=1, dropout_rate=None):
    model = Sequential()
    if num_conv_layers >= 1:
        model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
        model.add(MaxPooling2D((2, 2)))
    if num_conv_layers >= 2:
        model.add(Conv2D(64, (3, 3), activation='relu'))
        model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    if dropout_rate is not None:
        model.add(Dropout(dropout_rate))
    for _ in range(num_dense_layers - 1):
        model.add(Dense(128, activation='relu'))
        if dropout_rate is not None:
            model.add(Dropout(dropout_rate))
    model.add(Dense(10, activation='softmax'))
    return model

# Task: Modify CNN Architecture
print("\nExperimenting with CNN Architecture:")
architectures = [(1, 1), (2, 1), (1, 2)] # (num_conv_layers, num_dense_layers)
for conv_layers, dense_layers in architectures:
    print(f"\nArchitecture: {conv_layers} Conv Layers, {dense_layers} Dense Layers")
    model = create_cnn_model(num_conv_layers=conv_layers, num_dense_layers=dense_layers)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1, verbose=0)
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"Test Accuracy: {accuracy:.4f}")

# Task: Train with Different Epochs
print("\nTraining with Different Epochs:")
epochs_list = [2, 5, 10, 20]
histories = {}
for epochs in epochs_list:
    print(f"\nTraining for {epochs} epochs:")
    model = create_cnn_model() # Use a default architecture
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train, y_train, epochs=epochs, batch_size=32, validation_split=0.1, verbose=0)
    histories[epochs] = history
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"Test Accuracy: {accuracy:.4f}")

# Plot training and validation accuracy/loss for different epochs
plt.figure(figsize=(12, 6))
for epochs, history in histories.items():
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label=f'Train (Epochs: {epochs})')
    plt.plot(history.history['val_accuracy'], label=f'Validation (Epochs: {epochs})')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy vs. Epochs')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label=f'Train (Epochs: {epochs})')
    plt.plot(history.history['val_loss'], label=f'Validation (Epochs: {epochs})')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss vs. Epochs')
    plt.legend()
plt.tight_layout()
plt.show()

# Task: Plot Confusion Matrix as a Heatmap with Percentages
model_for_cm = create_cnn_model() # Create a fresh model for this task
model_for_cm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_for_cm.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1, verbose=0)
y_pred_probabilities = model_for_cm.predict(x_test)
y_pred_classes = np.argmax(y_pred_probabilities, axis=1)
cm = confusion_matrix(y_test, y_pred_classes)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt=".2%", cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix (Percentages)")
plt.show()

# Task: Visualize Predictions
print("\nVisualizing Predictions:")
num_samples_to_visualize = 10
random_indices = np.random.choice(len(x_test), num_samples_to_visualize, replace=False)
plt.figure(figsize=(15, 5))
for i, index in enumerate(random_indices):
    plt.subplot(1, num_samples_to_visualize, i + 1)
    plt.imshow(x_test[index].reshape(28, 28), cmap='gray')
    predicted_label = np.argmax(model_for_cm.predict(x_test[index].reshape(1, 28, 28, 1)), axis=1)[0]
    actual_label = y_test[index]
    plt.title(f"P: {predicted_label}\nA: {actual_label}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Task: Experiment with Dropout Layers
print("\nExperimenting with Dropout Layers:")
dropout_rates = [0.2, 0.5]
for rate in dropout_rates:
    print(f"\nDropout Rate: {rate}")
    model_dropout = create_cnn_model(dropout_rate=rate)
    model_dropout.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history_dropout = model_dropout.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1, verbose=0)
    loss_dropout, accuracy_dropout = model_dropout.evaluate(x_test, y_test, verbose=0)
    print(f"Test Accuracy with Dropout: {accuracy_dropout:.4f}")

# Task: Try Different Optimizers
print("\nTrying Different Optimizers:")
optimizers = {'Adam': Adam(), 'SGD': SGD(learning_rate=0.01), 'RMSprop': RMSprop(learning_rate=0.001)}
for name, optimizer in optimizers.items():
    print(f"\nOptimizer: {name}")
    model_optimizer = create_cnn_model() # Use a default architecture
    model_optimizer.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history_optimizer = model_optimizer.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1, verbose=0)
    loss_optimizer, accuracy_optimizer = model_optimizer.evaluate(x_test, y_test, verbose=0)
    print(f"Test Accuracy with {name}: {accuracy_optimizer:.4f}")

# Task: Analyze precision, recall, and F1-score for each class
model_for_report = create_cnn_model() # Create a fresh model for the report
model_for_report.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_for_report.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1, verbose=0)
y_pred_probabilities_report = model_for_report.predict(x_test)
y_pred_classes_report = np.argmax(y_pred_probabilities_report, axis=1)
report = classification_report(y_test, y_pred_classes_report)
print("\nClassification Report (Analyzing Precision, Recall, F1-score for each class):\n")
print(report)

Training set shape: (60000, 28, 28)
Test set shape: (10000, 28, 28)
Normalized image shape: (28, 28, 1)

Experimenting with CNN Architecture:

Architecture: 1 Conv Layers, 1 Dense Layers
